In [1]:
import pandas as pd
import sqlite3

# Step 1: Initialize local SQL Environment

# 1. Load the newly merged dataset
df_pipeline = pd.read_csv('../dataset/processed/meridian_pipeline_clean.csv')

# 2. Create a temporary SQLite database
conn = sqlite3.connect(':memory:')

# 3. Push the dataframe into a SQL table
df_pipeline.to_sql('pipeline', conn, index=False, if_exists='replace')

# 4. Helper function to run SQL queries easily
def run_query(query):
    return pd.read_sql_query(query, conn)

print("SQL database successfully initialized!")

SQL database successfully initialized!


In [ ]:
# Step 2: Find out which territories are over-resourced or under-resourced

query_territory = """
SELECT
    regional_office,
    COUNT(DISTINCT sales_agent) as total_headcount,
    COUNT(opportunity_id) as total_deal_volume,
    SUM(CASE WHEN deal_stage = 'Won' THEN close_value ELSE 0 END) as total_won_revenue,
    ROUND(CAST(SUM(CASE WHEN deal_stage = 'Won' THEN 1 ELSE 0 END) AS FLOAT) / NULLIF(COUNT(CASE WHEN deal_stage IN ('Won', 'Lost') THEN opportunity_id END), 0) * 100, 2) as historical_win_rate_pct
FROM pipeline
GROUP BY regional_office
ORDER BY total_won_revenue DESC;
"""

run_query(query_territory)

,regional_office,total_headcount,total_deal_volume,total_won_revenue,historical_win_rate_pct
0,West,10,2997,3568647.0,63.94
1,Central,10,3512,3346293.0,62.56
2,East,10,2291,3090594.0,63.02


In [5]:
# Step 3: Find reps who are hoarding massive amounts of "In Progress" deals but have historically terrible win rates

query_utilization = """
SELECT 
    sales_agent,
    regional_office,
    COUNT(CASE WHEN deal_stage IN ('In Progress', 'Prospecting', 'Engaging') THEN opportunity_id END) as active_inflated_pipeline,
    ROUND(CAST(SUM(CASE WHEN deal_stage = 'Won' THEN 1 ELSE 0 END) AS FLOAT) / 
          NULLIF(COUNT(CASE WHEN deal_stage IN ('Won', 'Lost') THEN opportunity_id END), 0) * 100, 2) as historical_win_rate_pct
FROM pipeline
GROUP BY sales_agent, regional_office
HAVING active_inflated_pipeline > 0
ORDER BY active_inflated_pipeline DESC
LIMIT 10;
"""

run_query(query_utilization)

,sales_agent,regional_office,active_inflated_pipeline,historical_win_rate_pct
0,Darcel Schlecht,Central,194,63.11
1,Anna Snelling,Central,112,61.90
2,Vicki Laflamme,West,104,63.69
3,Kary Hendrixson,West,103,62.39
4,Versie Hillebrand,Central,97,66.67
5,Kami Bicknell,West,90,63.97
6,Zane Levy,West,88,61.69
7,Marty Freudenburg,Central,87,62.89
8,Cassey Cress,East,85,62.45
9,Gladys Colclough,Central,85,58.19


In [7]:
# Step 3: Because Darcel Schlecht is carrying a massive capacity score, let's find out id everyone in his regional office is burning out or if it's just him

query_burnout = """
SELECT 
    sales_agent,
    manager,
    regional_office,
    COUNT(opportunity_id) as assigned_deals,
    ROUND(CAST(SUM(CASE WHEN deal_stage = 'Won' THEN 1 ELSE 0 END) AS FLOAT) / 
          NULLIF(COUNT(CASE WHEN deal_stage IN ('Won', 'Lost') THEN opportunity_id END), 0) * 100, 2) as win_rate_pct
FROM pipeline
WHERE regional_office = (
    SELECT regional_office FROM pipeline WHERE sales_agent = 'Darcel Schlecht' LIMIT 1
)
GROUP BY sales_agent, manager, regional_office
ORDER BY assigned_deals DESC;
"""

run_query(query_burnout)

,sales_agent,manager,regional_office,assigned_deals,win_rate_pct
0,Darcel Schlecht,Melvin Marxen,Central,747,63.11
1,Anna Snelling,Dustin Brinkmann,Central,448,61.90
2,Versie Hillebrand,Dustin Brinkmann,Central,361,66.67
3,Jonathan Berthelot,Melvin Marxen,Central,345,64.77
4,Gladys Colclough,Melvin Marxen,Central,317,58.19
5,Lajuana Vencill,Dustin Brinkmann,Central,311,54.98
6,Marty Freudenburg,Melvin Marxen,Central,281,62.89
7,Moses Frase,Dustin Brinkmann,Central,260,66.15
8,Niesha Huffines,Melvin Marxen,Central,239,60.00
9,Cecily Lampkin,Dustin Brinkmann,Central,203,66.88
